# 12. Verification Completion & FE-v2 Pipeline (`<= 120` Features) on Google Colab GPU

Notebook tuân thủ đúng quy trình **Local (code + GitHub) ↔ Colab (GPU/RAM trên `/content/` SSD + checkpoint từng bước về Drive)**:
- **Code mới nhất** luôn `git pull` trực tiếp từ `https://github.com/AnhQun18/Propensity.git` (`main`).
- **Dữ liệu cache & feature mart (`mbd_colab_bundle.zip` + `panel_fe_v2_candidates.parquet`)** giải nén vào `/content/mbd_work` (SSD local) để tránh nghẽn Drive FUSE.
- **Checkpoint từng bước** ghi trực tiếp kết quả gọn nhẹ ra `/content/drive/MyDrive/mbd_colab_outputs/` ngay sau mỗi bước:
  1. `Group 2.4`: `FE-v1` OOF Permutation Importance (139 biến, auto-GPU).
  2. `Group 3`: Multi-seed (`L5c_cb` 3 seeds, `L5c_lgbm` 5 seeds, `B_catboost_TGD_tuned` 5 seeds).
  3. `FE-v2` (≤ 120 biến, **bản rút gọn 4 ứng viên chính trên Val `seed=42` + 5 seeds Locked Test**).

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

!nvidia-smi || true
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "catboost", "lightgbm", "xgboost", "scikit-learn", "pandas", "pyarrow", "scipy", "duckdb", "matplotlib"
])

In [ ]:
# 1) Giải nén cache/data từ Drive vào SSD /content/mbd_work
# 2) Đồng bộ code mới nhất từ GitHub (origin main)
DRIVE_DIR = Path("/content/drive/MyDrive")
SAVE_DIR = DRIVE_DIR / "mbd_colab_outputs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BUNDLE_ZIP = DRIVE_DIR / "mbd_colab_bundle.zip"
PATCH_ZIP = DRIVE_DIR / "mbd_colab_patch_core.zip"
CAND_PARQUET_DRIVE = DRIVE_DIR / "panel_fe_v2_candidates.parquet"
WORK_DIR = Path("/content/mbd_work")

assert BUNDLE_ZIP.exists(), f"Không thấy {BUNDLE_ZIP}"
WORK_DIR.mkdir(parents=True, exist_ok=True)

if not (WORK_DIR / "outputs" / "baseline_moscow_v1" / "dialog_emb_moscow.npz").exists():
    print("Extracting mbd_colab_bundle.zip ...")
    !unzip -q -o "{BUNDLE_ZIP}" -d "{WORK_DIR}"

if PATCH_ZIP.exists() and not (WORK_DIR / "outputs" / "fe_v2" / "panel_fe_v2_candidates.parquet").exists():
    print("Extracting mbd_colab_patch_core.zip ...")
    !unzip -q -o "{PATCH_ZIP}" -d "{WORK_DIR}"
elif CAND_PARQUET_DRIVE.exists() and not (WORK_DIR / "outputs" / "fe_v2" / "panel_fe_v2_candidates.parquet").exists():
    (WORK_DIR / "outputs" / "fe_v2").mkdir(parents=True, exist_ok=True)
    !cp "{CAND_PARQUET_DRIVE}" "{WORK_DIR}/outputs/fe_v2/panel_fe_v2_candidates.parquet"

# Luôn kéo code + configs mới nhất từ GitHub repo (không đè lên cache outputs đã giải nén)
%cd /content/mbd_work
!rm -rf /tmp/prop_repo && git clone --depth 1 https://github.com/AnhQun18/Propensity.git /tmp/prop_repo
!cp -r /tmp/prop_repo/src /tmp/prop_repo/configs /tmp/prop_repo/docs /tmp/prop_repo/tests /content/mbd_work/
!ls -lh outputs/fe_v2/panel_fe_v2_candidates.parquet outputs/baseline_moscow_v1/dialog_emb_moscow.npz

In [ ]:
# Bước 1: Group 2.4 FE-v1 Permutation Importance (139 biến) -> Checkpoint ra Drive
!python -B -X utf8 src/scripts/verify_group2_utc_and_pi.py
!mkdir -p "{SAVE_DIR}/verification_audit" && cp -f outputs/verification_audit/*.csv "{SAVE_DIR}/verification_audit/"

In [ ]:
# Bước 2: Group 3 Multi-seed (L5c_cb 3 seeds, L5c_lgbm 5 seeds, Protocol B 5 seeds) -> Checkpoint ra Drive
!python -B -X utf8 src/scripts/verify_group3_multiseed.py
!cp -f outputs/verification_audit/*.csv "{SAVE_DIR}/verification_audit/"

In [ ]:
# Bước 3A: FE-v2 Stage A + Stage B (Chọn <= 120 biến trên Val + Screen 4 ứng viên seed=42) -> Checkpoint ra Drive
!python -B -X utf8 src/scripts/run_fe_v2_pipeline.py --stage val_only --val-seeds 42
!mkdir -p "{SAVE_DIR}/fe_v2" && cp -f outputs/fe_v2/*.json outputs/fe_v2/*.csv "{SAVE_DIR}/fe_v2/"

In [ ]:
# Bước 3B: FE-v2 Stage C (Locked Test 5 seeds + Bootstrap Holm/BH + Calibration Slope/Intercept + Sanity) -> Sync Drive
!python -B -X utf8 src/scripts/run_fe_v2_pipeline.py --stage test_confirm
!cp -rf outputs/fe_v2/* "{SAVE_DIR}/fe_v2/" || true
!zip -q -r "{DRIVE_DIR}/mbd_colab_results.zip" outputs/verification_audit/*.csv outputs/verification_audit/*.json outputs/fe_v2/*.csv outputs/fe_v2/*.json outputs/fe_v2/predictions
!ls -lh "{DRIVE_DIR}/mbd_colab_results.zip"